# Runbook mensuel — Pipeline Data & Décision Marketing

**Awalé Boissons**

**Objectif :** produire chaque mois une analyse reproductible du budget marketing, des ventes et de la Customer Voice, puis préparer une proposition testable d'allocation budgétaire en moins d'une heure, sans data engineer.


## 1. Préparer les nouvelles données

Déposer les exports du nouveau mois sans modifier les historiques.

```
data/
  raw/
    awale_boissons_starter_dataset.xlsx
```

**Règle :** une valeur manquante reste manquante. Une journée absente n'est jamais transformée automatiquement en zéro vente.


## 2. Contrôler DuckDB

Vérifier que les tables RAW attendues sont présentes et que le nouveau périmètre est bien chargé.

```bash
cd <racine du projet>
duckdb data/awale.duckdb
SHOW TABLES;
```


## 3. Exécuter dbt

Lancer le pipeline puis les tests.

```bash
cd dbt
dbt run
dbt test
```

**Critère de passage :** aucune erreur et aucun avertissement.


## 4. Contrôles de qualité des données

| Domaine | Contrôles essentiels |
|---|---|
| Ventes | Doublons, dates, retours, unités/CA négatifs, jours manquants, couverture. |
| Marketing | Campagnes vs media plan vs facturé ; divergences visibles ; aucune vérité universelle choisie arbitrairement. |
| WhatsApp | Montants manquants, order_ref répétés, parsing des articles, produits inconnus. |
| Social | Doublons, unicité de comment_id, prédictions non NULL, volume source = volume prédit. |

**Important :** ne jamais construire un catalogue de prix à partir de `amount_fcfa` / `quantity`. Les montants peuvent contenir des anomalies ou agrégations.


## 4bis. Si un test échoue à 8h, le jour du run (rapport dû à midi)


Principe général : **isoler et documenter, ne jamais livrer un chiffre non contrôlé sans avertissement.** Détail par domaine :

| Domaine | Réaction en moins de 2h |
|---|---|
| Ventes (doublons, dates, unités/CA négatifs inattendus) | Isoler les lignes en échec (`dbt test --store-failures` ou requête manuelle sur le modèle concerné). Si l'anomalie est isolable et documentable avant midi, exclure les lignes fautives avec un flag de qualité et continuer. Sinon, publier le rapport avec un bandeau explicite « CA du mois non entièrement validé » plutôt que des chiffres non contrôlés. |
| Marketing (écart export campagne / media plan anormal) | Vérifier en priorité une erreur de saisie manuelle sur les lignes radio/influenceurs (elles sont tapées à la main). Si non résolu avant midi, laisser l'écart affiché tel quel dans le dashboard (déjà prévu par construction) plutôt que de choisir arbitrairement une source. |
| WhatsApp (téléphone non normalisé, montant négatif, order_ref anormal) | Isoler les commandes en échec et les exclure du calcul de CA livraison avec un compteur visible (`amount_missing`, `order_ref_repeated`). Ne jamais les supprimer silencieusement. |
| Social / IA (comment_id dupliqué, prédictions manquantes ou NULL) | Ne pas construire `mart_social_monthly` ce mois-ci plutôt que de le construire avec des trous. `run_pipeline.py` gère déjà ce cas : si `raw_social_comments_predictions_v2` n'est pas prête, il exclut automatiquement les modèles Customer Voice du run et le dashboard affiche l'avertissement correspondant. |

Si aucun contournement propre n'est possible avant midi : envoyer la note client avec la section concernée explicitement marquée comme indisponible ce mois-ci, plutôt que retarder tout le rapport ou publier un chiffre non fiable.

## 5. Exécuter et surveiller l'IA

- Le benchmark humain de 50 commentaires reste séparé de l'inférence complète : il sert à évaluer le modèle, pas à prétendre à un fine-tuning.
- Comparer le nombre de commentaires source et le nombre de prédictions.
- Vérifier l'unicité de `comment_id` et l'absence de NULL sur `language`, `sentiment`, `theme`, `product` et `spam`.
- Conserver version du modèle, résultats d'évaluation, erreurs, volume, temps d'exécution, coût éventuel et limites.
- Ne jamais laisser le modèle inventer des chiffres ou des faits absents des données.


## 6. Vérifier le dashboard — 4 blocs maximum

| Bloc | Question |
|---|---|
| 1. Où va l'argent ? | Répartition du budget et divergences avec le plan. |
| 2. Ventes | Évolution du CA et des unités, avec couverture et jours manquants. |
| 3. Customer Voice | Sentiment, thèmes et produits dans les commentaires. |
| 4. What to do next | Proposition de budget testable, conditions de mesure et limites. |


## 7. Chaîne de décision

```
Faits observés → qualité des données → observations → signaux → limites → hypothèses → test → décision
```

**Exemple :** beaucoup de commentaires négatifs sur le prix constitue un signal de Customer Voice. Cela ne démontre pas qu'un canal marketing en est la cause ni qu'un changement de budget améliorera les ventes.


## 8. Règles méthodologiques non négociables

- Ne pas transformer les valeurs manquantes en zéros.
- Ne pas inventer de produit, format ou prix.
- Ne pas présenter une association canal → ventes comme une attribution causale.
- Ne pas utiliser `mart_channel_performance_monthly` pour attribuer le CA aux canaux : le CA mensuel total y est répété par canal.
- Ne pas transformer CPC, CPM, impressions ou clics en preuve d'efficacité commerciale.
- Ne pas présenter l'allocation de 15 M FCFA comme un ROI causal.
- Documenter les conditions d'instrumentation nécessaires pour tester les canaux faiblement mesurés.


## 9. Proposition de séquence pour les 15 M FCFA

| Canal | Budget proposé |
|---|---|
| Meta | 5 550 000 FCFA |
| TikTok | 3 100 000 FCFA |
| Radio | 2 000 000 FCFA |
| Google | 1 900 000 FCFA |
| Influenceurs | 1 450 000 FCFA |
| Activation terrain | 1 000 000 FCFA |
| **TOTAL** | **15 000 000 FCFA** |

Ce budget est calculé (`mart_budget_recommendation_15m`) à partir d'un socle d'instrumentation par canal et d'un score = part de dépense observée × qualité d'evidence ; il se recalcule à chaque run et n'est donc pas figé d'un mois sur l'autre. Ce n'est pas un classement causal des canaux.


## 10. Conditions de test par canal

| Canal | Condition de mesure |
|---|---|
| Meta | Impressions, clics et conversion mesurable. |
| TikTok | Impressions, clics et conversion mesurable. |
| Google | Clics et conversion mesurable. |
| Radio | Code, numéro ou mécanisme de suivi dédié. |
| Influenceurs | Lien, code ou mécanisme de suivi dédié. |
| Activation terrain | Réconcilier facturation et dépenses campagne avant extrapolation. |


## 11. Temps mensuel — mesuré, pas estimé

`run_pipeline.py` (sans `--skip-ai`) a été exécuté de bout en bout le 2026-09-18. Les temps
ci-dessous sont mesurés sur ce run, sauf mention contraire :

| Étape | Temps | Nature |
|---|---|---|
| Préparation fichiers | 10 min | Estimé — revue humaine, non mesurable par un run |
| Ingestion (`load_raw.py`) | **2,5 s mesurés** | — |
| dbt run (1<sup>re</sup> passe, hors IA) | **7 s mesurés** | — |
| Export texte IA | **1,1 s mesuré** | — |
| Inférence IA (modèle local, CPU, incrémentale) | ~45 min pour un mois type (~470 nouveaux commentaires, débit de 5,55 s/commentaire mesuré sur un échantillon de 120) | Mesuré et extrapolé |
| Rechargement prédictions | **2,3 s mesurés** | — |
| dbt run (2<sup>e</sup> passe, complet) | **13,8 s mesurés** | — |
| dbt test | **10,2 s mesurés** | — |
| Contrôles qualité | 10 min | Estimé — revue humaine |
| Contrôle dashboard | 5 min | Estimé — revue humaine |
| **TOTAL (mois régulier)** | **≈ 70 min — toujours au-dessus de l'heure cible du brief, mais loin des ~302 min d'avant l'optimisation (40 min de socle fixe estimé + 262 min d'IA mesurés sur l'historique complet)** | |

Constat : dbt/DuckDB ne coûtent quasiment rien (~35 s cumulées, mesurées) — tout le temps du
cycle mensuel vient de l'IA (~45 min) et de la revue humaine (25 min, non compressible).

`classify_comments_hybrid.py` est incrémental depuis le 2026-09-18 : il ne classe que les
commentaires absents de `ai/evaluation/social_comments_predictions_v2_full.csv`, jamais tout
l'historique. Validé deux fois : retrait de 15 puis de 10 commentaires du fichier, relance
(dont une fois via `run_pipeline.py` complet), résultats bit-à-bit identiques aux prédictions
d'origine (décodage déterministe, `do_sample=False`).

La cible « moins d'une heure » du brief n'est donc pas encore strictement atteinte (10 min prépa +
45 min IA + 10 min qualité + 5 min dashboard, le reste étant négligeable). Nuance : `dbt run`,
`dbt test` et l'inférence IA ne demandent pas d'attention active — seuls « Préparation fichiers »,
« Contrôles qualité » et « Contrôle dashboard » (25 min) requièrent la présence du community
manager ; le reste tourne sans supervision. Piste technique encore ouverte pour réduire le temps
de calcul lui-même : diminuer `max_new_tokens` (80, largement supérieur à la taille d'un JSON de
réponse) ou augmenter `BATCH_SIZE`. Non implémenté à ce stade.

**Exception : le tout premier run** (aucune prédiction existante) doit classer les 2 831
commentaires de l'historique et mesure ~262 min (voir `docs/ai_documentation.ipynb` §8). À
anticiper avant la première mise en production, pas à chaque cycle mensuel.


## 12. Commandes de fin de run

```bash
cd <racine du projet>
python run_pipeline.py
streamlit run app/app.py
```

`run_pipeline.py` enchaîne ingestion, dbt run, export IA, inférence, rechargement des
prédictions, dbt run complet et dbt test (voir README §14.3). Utiliser `--skip-ai` pour un
run de test rapide sans relancer l'inférence.

**Livrable mensuel :** pipeline exécuté et testé, dashboard actualisé, contrôles qualité vérifiés, proposition budgétaire documentée et limites explicitement signalées.
